# OR-TS in ten minutes

**Odds-Ratio Thompson Sampling** is Thompson sampling for A/B tests and bandits whose baseline keeps moving. It fits, once per batch, an ordinary reference-coded logistic regression on the batch's counts with a fresh intercept, and carries across batches only the joint posterior of the **contrasts** (log odds ratios), never the level.

This notebook shows, in order: what the state remembers, one update cycle, what happens when the platform shifts, how OR-TS compares with Beta-Bernoulli and full logistic Thompson sampling, how to check the assumption on your own logs, and a default stopping rule.

Paper: S. Kim (2026), *Odds-Ratio Thompson Sampling: A Specification and Design Guide for Contrast-Based Multi-Armed Bandits*. Code: [github.com/sulgik/orts](https://github.com/sulgik/orts).

In [ ]:
%pip -q install orts matplotlib
import numpy as np, matplotlib.pyplot as plt
from orts import LogisticBandit, TSPar, DiscountedTSPar, diagnostics as dg
import orts; print("orts", orts.__version__)

## 1. What the dashboard shows, and what OR-TS remembers

One synthetic experiment, three arms, forty batches. The common level follows a random walk; the contrasts are constant. Left: the event rates every dashboard plots. Right: the same batches in contrast coordinates.

In [ ]:
rng = np.random.default_rng(30)
T, n = 40, 200_000
beta = np.array([0.0, 0.35, 0.70])                      # true log-odds contrasts vs arm A
alpha = np.log(0.01 / 0.99) + np.cumsum(rng.normal(0, 0.16, T)); alpha -= alpha.mean() - np.log(0.01 / 0.99)
p = 1 / (1 + np.exp(-(alpha[:, None] + beta[None, :])))
phat = rng.binomial(n, p) / n
bhat = np.log(phat / (1 - phat)) - np.log(phat[:, :1] / (1 - phat[:, :1]))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for k, lab in enumerate(["A (ref)", "B", "C"]):
    ax[0].plot(100 * phat[:, k], label=lab)
ax[0].set(title="observed event rate (%)", xlabel="batch"); ax[0].legend(frameon=False)
for k in (1, 2):
    ax[1].plot(bhat[:, k]); ax[1].axhline(beta[k], ls="--", color="0.6")
ax[1].set(title="estimated contrast vs A", xlabel="batch", ylim=(-0.1, 1.0)); plt.show()

## 2. One update cycle (Algorithm 1)

Hand the batch's counts to `update` (R1: fit with a fresh flat intercept; R2: keep the contrast posterior, discard the intercept). Then `win_prop` (A1: draw contrasts, score the reference 0, find winners; A2: winner shares are the next allocation).

In [ ]:
bandit = LogisticBandit(reference="A")
bandit.update({"A": [30000, 300], "B": [30000, 330], "C": [30000, 290]})
print("state order (reference last):", bandit.action_list)
print("contrasts vs A  {arm: (mean, sd)}:", {a: (round(m, 3), round(s, 3)) for a, (m, s) in bandit.contrasts().items()})
print("level of this batch (discarded next time):", round(bandit.level, 3))
print("next allocation:", bandit.win_prop(draw=50_000, rng=np.random.default_rng(0)))

## 3. The platform shifts

Every rate halves. A per-arm state now holds three obsolete numbers; OR-TS's contrasts did not move, and the new level is fitted from the new batch alone.

In [ ]:
bandit.update({"A": [30000, 150], "B": [30000, 165], "C": [30000, 145]})
print("contrasts after the shift:", {a: round(m, 3) for a, (m, _) in bandit.contrasts().items()})
print("new level:", round(bandit.level, 3))
print("next allocation:", bandit.win_prop(draw=50_000, rng=np.random.default_rng(0)))

## 4. Three bets, one shock

Beta-TS bets that every arm's rate is fixed; Full-TS makes the same bet in logistic coordinates; OR-TS bets only that the contrasts are. A common shock, redrawn every batch, is the disturbance the contrast bet allows and the other two do not.

In [ ]:
K, N, T = 5, 100_000, 30
contrasts = np.array([0.0, 0.05, 0.10, 0.15, 0.20]); arms = [f"arm{i}" for i in range(K)]
specs = {"Beta-TS": (TSPar, {}), "Full-TS": (LogisticBandit, {"odds_ratios_only": False}), "OR-TS": (LogisticBandit, {})}
rng = np.random.default_rng(100); levels = np.log(0.03 / 0.97) + rng.normal(0, 0.3, T)
share = {}
for name, (cls, kw) in specs.items():
    pol, alloc, s = cls(), {a: 1 / K for a in arms}, []
    for t in range(T):
        obs = {}
        for i, a in enumerate(arms):
            m = int(N * alloc[a]); pr = 1 / (1 + np.exp(-(levels[t] + contrasts[i])))
            obs[a] = [m, int(rng.binomial(m, pr))] if m > 0 else [0, 0]
        pol.update(obs, **kw); alloc = pol.win_prop(draw=20_000, rng=rng); s.append(alloc["arm4"])
    share[name] = s
for name, s in share.items(): plt.plot(range(1, T + 1), s, label=name)
plt.ylabel("share of traffic on the best arm"); plt.xlabel("batch"); plt.legend(frameon=False); plt.show()

## 5. Is the assumption holding on *your* data?

The assumption is that the level moves and the contrasts do not. From per-batch counts alone, `orts.diagnostics` computes each batch's level and contrasts with their sampling variances, the excess variance beyond noise, the ratio $R = \text{excess sd}(\alpha)/\text{excess sd}(\beta)$, and the decay that any contrast drift implies.

Replace the synthetic `batches` below with your own list of `{arm: [exposures, events]}` dicts, one per batch, in time order.

In [ ]:
# ---- your data goes here: one dict per batch, e.g. read from a CSV with columns batch, arm, exposures, events
rng = np.random.default_rng(7)
batches = []
for t in range(60):
    level = -3.0 + rng.normal(0, 0.3)                    # a wandering level ...
    contrast = 0.15                                      # ... and a constant contrast
    n = 40_000
    batches.append({"control": [n, rng.binomial(n, 1 / (1 + np.exp(-level)))],
                    "treatment": [n, rng.binomial(n, 1 / (1 + np.exp(-(level + contrast))))]})

alphas, avars, betas, bvars = [], [], [], []
for obs in batches:
    (a, av), c = dg.batch_contrasts(obs, reference="control")
    alphas.append(a); avars.append(av); betas.append(c["treatment"][0]); bvars.append(c["treatment"][1])

print(f"R = {dg.level_contrast_ratio(alphas, avars, betas, bvars):.1f}   (>> 1: the level moves, the contrast does not)")
print(f"excess sd  level {dg.excess_sd(alphas, avars):.3f}   contrast {dg.excess_sd(betas, bvars):.4f}")
print(f"lag-1 autocorrelation  level {dg.lag1_autocorrelation(alphas):.2f}   contrast {dg.lag1_autocorrelation(betas):.2f}")

band = dg.sampling_band(bvars)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(np.array(alphas) - np.mean(alphas)); ax[0].set(title="level (demeaned)", xlabel="batch")
b = np.array(betas); ax[1].plot(b - b.mean()); ax[1].fill_between(range(len(b)), -band, band, alpha=0.2)
ax[1].set(title="contrast (demeaned) and its ±2 sampling band", xlabel="batch"); plt.show()

# the decay that the measured contrast drift implies for a running OR-TS state
running = LogisticBandit(reference="control")
for obs in batches: running.update(obs)
print("implied decay lambda:", round(running.implied_decay(dg.excess_sd(betas, bvars)), 4), "(0 means: carry everything)")

## 6. A default stopping rule

`win_prop` gives each arm's posterior probability of being best and `expected_loss` the expected loss of committing to it now, in log-odds units. A workable default: drop an arm whose probability stays below 1% for three batches; stop when the leader exceeds 95% and its expected loss is below what you are willing to forgo. Because both read the contrast posterior, neither moves when the level moves.

In [ ]:
rng = np.random.default_rng(11)
truth = {"control": 0.030, "v1": 0.031, "v2": 0.033}
bandit = LogisticBandit(reference="control"); alloc = {a: 1 / 3 for a in truth}; active = list(truth); below = {a: 0 for a in truth}
for t in range(1, 41):
    shift = rng.normal(0, 0.25)
    obs = {a: [int(20000 * alloc[a]), 0] for a in active}
    for a in active:
        pr = truth[a] * np.exp(shift) / (1 - truth[a] + truth[a] * np.exp(shift)); obs[a][1] = int(rng.binomial(obs[a][0], pr))
    bandit.update(obs, remove_not_observed=True)
    alloc = bandit.win_prop(active, draw=20_000, rng=rng); loss = bandit.expected_loss(active, draw=20_000, rng=rng)
    leader = max(alloc, key=alloc.get)
    for a in active: below[a] = below[a] + 1 if alloc[a] < 0.01 else 0
    for a in [a for a in active if below[a] >= 3 and a != leader]:
        active.remove(a); print(f"batch {t}: drop {a}")
    if alloc[leader] >= 0.95 and loss[leader] <= 0.02:
        print(f"batch {t}: stop; {leader} P(best)={alloc[leader]:.3f}, expected loss {loss[leader]:.4f}"); break

## Where to go next

- `LogisticBandit.from_beta_posteriors({arm: (a, b)})` warm-starts from a running Beta-Bernoulli service.
- `update(obs, decay=λ)` and `win_prop(aggressive=γ, floor=f)` are the two controls of the paper's Section 5; `implied_decay` sets λ from measured drift.
- Keep expected events per arm per batch above ten; the lever is the cycle length, not the method.